In [1]:
import pandas as pd
import os
import re
import pickle

## Trying and Testing

Membership matrix

In [ ]:
# n = len(nodes_1992) # no. of papers (rows)
# m = len(partitions) # no. of communities (columns)

In [ ]:
# # Membership matrix for 1992

# # n --> nodes; m --> communities 
# # n = 1051; m = 681

# membership_1992 = np.zeros((n, m), dtype=int)

# for paperId, x in paperId_map.items():
#     y = community_map_1992.get(paperId)
#     # print(y)
#     membership_1992[x][y] = 1

# print(membership_1992.sum())

# # Membership matrix for 1992 created

1051


Centrality Tensor

In [ ]:
# Creating a centrality tensor for 1992

# Centrality tensor is a completely different matrix from membership matrix. 
# Hence a slight modification is that, instead of author i,
# I will be using the paperId itself and try implementing COIN model.

# centrality_tensor = np.zeros((n, m, m), dtype=int)

# for citing, cited in g_directed_1992.edges():
#     paperId = paperId_map.get(cited)
#     u = int(community_map_1992.get(cited))
#     v = int(community_map_1992.get(citing))

#     centrality_tensor[paperId, u, v] += 1


In [ ]:
# print(centrality_tensor.sum(), len(g_directed_1992.edges()))

535 535


Impact Matrix

In [ ]:
# numerator = np.einsum('xu,xuv->uv', membership_1992, centrality_tensor)

In [ ]:
# numerator.shape

(681, 681)

In [ ]:
# community_sizes = membership_1992.sum(axis=0)
# community_sizes.shape

(681,)

In [ ]:
# # Impact Matrix

# J = numerator / community_sizes[:, None]
# J.shape

(681, 681)

In [ ]:
# with np.printoptions(threshold=np.inf):
#     print(J)

[[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.   

# Storing year-wise abstract data into parquet files

In [26]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\yadne\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\yadne\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yadne\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\yadne\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [27]:
# pre-processing function for SciBERT type models

def getTitleAndAbstract(text):

    # text = re.sub(r'\n', ' ', text)

    title = re.search(r'Title:\s*(.*)', text)
    sections = text.split('\\\\')
    abstract = sections[-2] if len(sections) > 2 else ""

    abstract_cleaned = re.sub(r'\n', ' ', abstract)

    return [f'{title.group(1).strip()}. {abstract_cleaned.strip()}'.strip()]



In [28]:
# pre-processing function for LDA type models

def lda_embeddings(text):

    # for excluding other unnecessary metadata like 
    # Journal Ref:, Date:, From:, etc. 
     
    title = re.search(r'Title:\s*(.*)', text)
    sections = text.split('\\\\')
    abstract = sections[-2] if len(sections) > 2 else ""
    abstract_cleaned = re.sub(r'\n', ' ', abstract)

    text = title.group(1).strip() + '\n' + abstract_cleaned.strip()

    # to lowercase
    text = text.lower()

    # remove latex math expressions
    text = re.sub(r'\$.*?\$', ' ', text)

    # remove latex commands
    text = re.sub(r'\\[A-Za-z]+', ' ', text)

    # remove punctuations
    text = re.sub(r'[^\w\s]', ' ', text)

    # remove standalone numbers
    text = re.sub(r'\b\d+\b', ' ', text)

    # remove underscores
    text = re.sub(r'_+', ' ', text)

    # correcting whitespaces
    text = re.sub(r'\s+', ' ', text)

    # tokenization
    tokens = word_tokenize(text)

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    return tokens



In [ ]:
# Storing Abstract data in structured format for different models 
# Ex: LDA (require processing), SciBERT (Entire sentences are required for understanding context), etc.

papers_data = []
directory = '../abstracts/1992'

for file in os.listdir(directory):
    filename = os.fsdecode(file)
    with open(f'{directory}/{filename}') as f:
    
        text = f.read()
        
        scibert_data = getTitleAndAbstract(text)
        # print(scibert_data)

        embeddings = lda_embeddings(text)
        # print(embeddings)

        paper_id = re.search(r'Paper:\s*hep-th/(\d{7})', text)
        title = re.search(r'Title:\s*(.*)', text)
        subject = re.search(r'Subj-class:\s*(.*)', text) 

        papers_data.append({
            'paper_id': paper_id.group(1).strip(),
            'year': 1992,
            'title': title.group(1).strip() if title else "",
            'subject': subject.group(1).strip() if subject else "",
            'scibert_data': scibert_data,
            'embeddings': embeddings
        })


df_1992 = pd.DataFrame(papers_data)
df_1992.head()


,paper_id,year,title,subject,scibert_data,embeddings
0,9201001,1992,Combinatorics of the Modular Group II: the Kon...,High Energy Physics - Theory; Quantum Algebra,[Combinatorics of the Modular Group II: the Ko...,"[combinatorics, modular, group, kontsevich, in..."
1,9201002,1992,Inomogeneous Quantum Groups as Symmetries of P...,,[Inomogeneous Quantum Groups as Symmetries of ...,"[inomogeneous, quantum, group, symmetry, phono..."
2,9201003,1992,"Intersection Theory, Integrable Hierarchies an...",,"[Intersection Theory, Integrable Hierarchies a...","[intersection, theory, integrable, hierarchy, ..."
3,9201004,1992,The Heterotic Green-Schwarz Superstring on an ...,,[The Heterotic Green-Schwarz Superstring on an...,"[heterotic, green, schwarz, superstring, super..."
4,9201005,1992,Ward Identities in Two-Dimensional String Theory,,[Ward Identities in Two-Dimensional String The...,"[ward, identity, two, dimensional, string, the..."


In [ ]:
df_1992.to_parquet('papers1992.parquet')

In [29]:
print(len(df_1992['scibert_data'][0][0]))

625


In [ ]:
# Storing Abstract data in structured format for different models 
# Ex: LDA (require processing), SciBERT (Entire sentences are required for understanding context), etc.

papers_data = []
directory = '../abstracts/1993'

for file in os.listdir(directory):
    filename = os.fsdecode(file)
    with open(f'{directory}/{filename}') as f:
    
        text = f.read()
        
        scibert_data = getTitleAndAbstract(text)
        # print(scibert_data)

        embeddings = lda_embeddings(text)
        # print(embeddings)

        paper_id = re.search(r'Paper:\s*hep-th/(\d{7})', text)
        title = re.search(r'Title:\s*(.*)', text)
        subject = re.search(r'Subj-class:\s*(.*)', text) 

        papers_data.append({
            'paper_id': paper_id.group(1).strip(),
            'year': 1993,
            'title': title.group(1).strip() if title else "",
            'subject': subject.group(1).strip() if subject else "",
            'scibert_data': scibert_data,
            'embeddings': embeddings
        })


df_1993 = pd.DataFrame(papers_data)

df_1993.to_parquet('abstract structured data/papers1993.parquet')

In [32]:
df_1993.head()

,paper_id,year,title,subject,scibert_data,embeddings
0,9301001,1993,Gonihedric String and Asymptotic Freedom,,[Gonihedric String and Asymptotic Freedom. Few...,"[gonihedric, string, asymptotic, freedom, natu..."
1,9301002,1993,Scaling behavior of quantum four-geometries,,[Scaling behavior of quantum four-geometries. ...,"[scaling, behavior, quantum, four, geometry, p..."
2,9301003,1993,Summing Over Inequivalent Maps in the String T...,,[Summing Over Inequivalent Maps in the String ...,"[summing, inequivalent, map, string, theory, i..."
3,9301004,1993,Large Order Behaviour of 2D Gravity Coupled to...,,[Large Order Behaviour of 2D Gravity Coupled t...,"[large, order, behaviour, gravity, coupled, ma..."
4,9301005,1993,"Duality, Marginal Perturbations and Gauging",,"[Duality, Marginal Perturbations and Gauging. ...","[duality, marginal, perturbation, gauging, stu..."


In [33]:
# Storing Abstract data in structured format for different models 
# Ex: LDA (require processing), SciBERT (Entire sentences are required for understanding context), etc.

papers_data = []
directory = '../abstracts/1994'

for file in os.listdir(directory):
    filename = os.fsdecode(file)
    with open(f'{directory}/{filename}') as f:
    
        text = f.read()
        
        scibert_data = getTitleAndAbstract(text)
        # print(scibert_data)

        embeddings = lda_embeddings(text)
        # print(embeddings)

        paper_id = re.search(r'Paper:\s*hep-th/(\d{7})', text)
        title = re.search(r'Title:\s*(.*)', text)
        subject = re.search(r'Subj-class:\s*(.*)', text) 

        papers_data.append({
            'paper_id': paper_id.group(1).strip(),
            'year': 1994,
            'title': title.group(1).strip() if title else "",
            'subject': subject.group(1).strip() if subject else "",
            'scibert_data': scibert_data,
            'embeddings': embeddings
        })


df_1994 = pd.DataFrame(papers_data)
df_1994.head()

,paper_id,year,title,subject,scibert_data,embeddings
0,9401001,1994,Remarks on the Collective Quantization of the ...,,[Remarks on the Collective Quantization of the...,"[remark, collective, quantization, skyrme, mod..."
1,9401002,1994,"Hard Thermal Loops, Static Response and the Co...",,"[Hard Thermal Loops, Static Response and the C...","[hard, thermal, loop, static, response, compos..."
2,9401003,1994,Local heterotic geometry in holomorphic coordi...,,[Local heterotic geometry in holomorphic coord...,"[local, heterotic, geometry, holomorphic, coor..."
3,9401004,1994,Vertex Operators are Not Closeable,,[Vertex Operators are Not Closeable. We prove ...,"[vertex, operator, closeable, prove, uncloseab..."
4,9401005,1994,Q-Exact Actions for BF Theories,,[Q-Exact Actions for BF Theories. The actions ...,"[exact, action, theory, action, classical, con..."


In [ ]:
df_1994.to_parquet('abstract structured data/papers1994.parquet')

In [36]:
# Storing Abstract data in structured format for different models 
# Ex: LDA (require processing), SciBERT (Entire sentences are required for understanding context), etc.

papers_data = []
directory = '../abstracts/1995'

for file in os.listdir(directory):
    filename = os.fsdecode(file)
    with open(f'{directory}/{filename}') as f:
    
        text = f.read()
        
        scibert_data = getTitleAndAbstract(text)
        # print(scibert_data)

        embeddings = lda_embeddings(text)
        # print(embeddings)

        paper_id = re.search(r'Paper:\s*hep-th/(\d{7})', text)
        title = re.search(r'Title:\s*(.*)', text)
        subject = re.search(r'Subj-class:\s*(.*)', text) 

        papers_data.append({
            'paper_id': paper_id.group(1).strip(),
            'year': 1995,
            'title': title.group(1).strip() if title else "",
            'subject': subject.group(1).strip() if subject else "",
            'scibert_data': scibert_data,
            'embeddings': embeddings
        })


df_1995 = pd.DataFrame(papers_data)
df_1995.head()

,paper_id,year,title,subject,scibert_data,embeddings
0,9501001,1995,Anomaly and Exotic Statistics in One Dimension,,[Anomaly and Exotic Statistics in One Dimensio...,"[anomaly, exotic, statistic, one, dimension, s..."
1,9501002,1995,Super-${\tau}_{3}$QED and the dimensional redu...,,[Super-${\tau}_{3}$QED and the dimensional red...,"[super, qed, dimensional, reduction, work, sup..."
2,9501003,1995,"BRST quantization of gauge theories like SL(2,...",,[BRST quantization of gauge theories like SL(2...,"[brst, quantization, gauge, theory, like, inne..."
3,9501004,1995,Solving general gauge theories on inner produc...,,[Solving general gauge theories on inner produ...,"[solving, general, gauge, theory, inner, produ..."
4,9501005,1995,"Steps towards Lattice Virasoro Algebras: su(1,1)",,[Steps towards Lattice Virasoro Algebras: su(1...,"[step, towards, lattice, virasoro, algebra, ex..."


In [37]:
df_1995.to_parquet('abstract structured data/papers1995.parquet')

In [38]:
# Storing Abstract data in structured format for different models 
# Ex: LDA (require processing), SciBERT (Entire sentences are required for understanding context), etc.

papers_data = []
directory = '../abstracts/1996'

for file in os.listdir(directory):
    filename = os.fsdecode(file)
    with open(f'{directory}/{filename}') as f:
    
        text = f.read()
        
        scibert_data = getTitleAndAbstract(text)
        # print(scibert_data)

        embeddings = lda_embeddings(text)
        # print(embeddings)

        paper_id = re.search(r'Paper:\s*hep-th/(\d{7})', text)
        title = re.search(r'Title:\s*(.*)', text)
        subject = re.search(r'Subj-class:\s*(.*)', text) 

        papers_data.append({
            'paper_id': paper_id.group(1).strip(),
            'year': 1996,
            'title': title.group(1).strip() if title else "",
            'subject': subject.group(1).strip() if subject else "",
            'scibert_data': scibert_data,
            'embeddings': embeddings
        })


df_1996 = pd.DataFrame(papers_data)
df_1996.head()

,paper_id,year,title,subject,scibert_data,embeddings
0,9601001,1996,Integrable Z_n-Chiral Potts Model: The Missing...,,[Integrable Z_n-Chiral Potts Model: The Missin...,"[integrable, chiral, potts, model, missing, ra..."
1,9601002,1996,Actions for QCD-like strings,,[Actions for QCD-like strings. We introduce a ...,"[action, qcd, like, string, introduce, random,..."
2,9601003,1996,D-brane actions,,[D-brane actions. Effective world-brane action...,"[brane, action, effective, world, brane, actio..."
3,9601004,1996,An Extreme Black Hole with Electric Dipole Moment,,[An Extreme Black Hole with Electric Dipole Mo...,"[extreme, black, hole, electric, dipole, momen..."
4,9601005,1996,An Extension of Distribution Theory Related to...,,[An Extension of Distribution Theory Related t...,"[extension, distribution, theory, related, gau..."


In [39]:
df_1996.to_parquet('abstract structured data/papers1996.parquet')

In [40]:
# Storing Abstract data in structured format for different models 
# Ex: LDA (require processing), SciBERT (Entire sentences are required for understanding context), etc.

papers_data = []
directory = '../abstracts/1997'

for file in os.listdir(directory):
    filename = os.fsdecode(file)
    with open(f'{directory}/{filename}') as f:
    
        text = f.read()
        
        scibert_data = getTitleAndAbstract(text)
        # print(scibert_data)

        embeddings = lda_embeddings(text)
        # print(embeddings)

        paper_id = re.search(r'Paper:\s*hep-th/(\d{7})', text)
        title = re.search(r'Title:\s*(.*)', text)
        subject = re.search(r'Subj-class:\s*(.*)', text) 

        papers_data.append({
            'paper_id': paper_id.group(1).strip(),
            'year': 1997,
            'title': title.group(1).strip() if title else "",
            'subject': subject.group(1).strip() if subject else "",
            'scibert_data': scibert_data,
            'embeddings': embeddings
        })


df_1997 = pd.DataFrame(papers_data)
df_1997.head()

,paper_id,year,title,subject,scibert_data,embeddings
0,9701001,1997,"Ricci flat metrics in various dimensions, depe...",,"[Ricci flat metrics in various dimensions, dep...","[ricci, flat, metric, various, dimension, depe..."
1,9701002,1997,Comment on ``Additional analytically exact sol...,,[Comment on ``Additional analytically exact so...,"[comment, additional, analytically, exact, sol..."
2,9701003,1997,On Generalized Quantum Deformations and Symmet...,,[On Generalized Quantum Deformations and Symme...,"[generalized, quantum, deformation, symmetry, ..."
3,9701004,1997,Large-Order Perturbation Theory in Infrared-Un...,,[Large-Order Perturbation Theory in Infrared-U...,"[large, order, perturbation, theory, infrared,..."
4,9701005,1997,Spontaneous Collapse of Supersymmetry,,[Spontaneous Collapse of Supersymmetry. It is ...,"[spontaneous, collapse, supersymmetry, shown, ..."


In [41]:
df_1997.to_parquet('abstract structured data/papers1997.parquet')

In [42]:
# Storing Abstract data in structured format for different models 
# Ex: LDA (require processing), SciBERT (Entire sentences are required for understanding context), etc.

papers_data = []
directory = '../abstracts/1998'

for file in os.listdir(directory):
    filename = os.fsdecode(file)
    with open(f'{directory}/{filename}') as f:
    
        text = f.read()
        
        scibert_data = getTitleAndAbstract(text)
        # print(scibert_data)

        embeddings = lda_embeddings(text)
        # print(embeddings)

        paper_id = re.search(r'Paper:\s*hep-th/(\d{7})', text)
        title = re.search(r'Title:\s*(.*)', text)
        subject = re.search(r'Subj-class:\s*(.*)', text) 

        papers_data.append({
            'paper_id': paper_id.group(1).strip(),
            'year': 1998,
            'title': title.group(1).strip() if title else "",
            'subject': subject.group(1).strip() if subject else "",
            'scibert_data': scibert_data,
            'embeddings': embeddings
        })


df_1998 = pd.DataFrame(papers_data)
df_1998.head()

,paper_id,year,title,subject,scibert_data,embeddings
0,9801001,1998,Forced Topological Nontrivial Field Configurat...,,[Forced Topological Nontrivial Field Configura...,"[forced, topological, nontrivial, field, confi..."
1,9801002,1998,Duality of Chiral N=1 Supersymmetric Gauge The...,,[Duality of Chiral N=1 Supersymmetric Gauge Th...,"[duality, chiral, supersymmetric, gauge, theor..."
2,9801003,1998,Effective interactions of a light gravitino,,[Effective interactions of a light gravitino. ...,"[effective, interaction, light, gravitino, rev..."
3,9801004,1998,Chiral anomaly and unambiguous rational terms,,[Chiral anomaly and unambiguous rational terms...,"[chiral, anomaly, unambiguous, rational, term,..."
4,9801005,1998,Quantum field theory can be UV finite if it is...,,[Quantum field theory can be UV finite if it i...,"[quantum, field, theory, finite, taken, effect..."


In [43]:
df_1998.to_parquet('abstract structured data/papers1998.parquet')

In [44]:
# Storing Abstract data in structured format for different models 
# Ex: LDA (require processing), SciBERT (Entire sentences are required for understanding context), etc.

papers_data = []
directory = '../abstracts/1999'

for file in os.listdir(directory):
    filename = os.fsdecode(file)
    with open(f'{directory}/{filename}') as f:
    
        text = f.read()
        
        scibert_data = getTitleAndAbstract(text)
        # print(scibert_data)

        embeddings = lda_embeddings(text)
        # print(embeddings)

        paper_id = re.search(r'Paper:\s*hep-th/(\d{7})', text)
        title = re.search(r'Title:\s*(.*)', text)
        subject = re.search(r'Subj-class:\s*(.*)', text) 

        papers_data.append({
            'paper_id': paper_id.group(1).strip(),
            'year': 1999,
            'title': title.group(1).strip() if title else "",
            'subject': subject.group(1).strip() if subject else "",
            'scibert_data': scibert_data,
            'embeddings': embeddings
        })


df_1999 = pd.DataFrame(papers_data)
df_1999.head()

,paper_id,year,title,subject,scibert_data,embeddings
0,9901001,1999,String Junctions and Their Duals in Heterotic ...,,[String Junctions and Their Duals in Heterotic...,"[string, junction, duals, heterotic, string, t..."
1,9901002,1999,Exact Absorption Probability in the Extremal S...,,[Exact Absorption Probability in the Extremal ...,"[exact, absorption, probability, extremal, six..."
2,9901003,1999,N=1 D=4 supermembrane in the coset approach,,[N=1 D=4 supermembrane in the coset approach. ...,"[supermembrane, coset, approach, supermembrane..."
3,9901004,1999,"Brane Waves, Yang-Mills theories and Causality",,"[Brane Waves, Yang-Mills theories and Causalit...","[brane, wave, yang, mill, theory, causality, p..."
4,9901005,1999,"Issues of Heterotic (0,2) Compactifications",,"[Issues of Heterotic (0,2) Compactifications. ...","[issue, heterotic, compactifications, brief, r..."


In [45]:
df_1999.to_parquet('abstract structured data/papers1999.parquet')

In [46]:
# Storing Abstract data in structured format for different models 
# Ex: LDA (require processing), SciBERT (Entire sentences are required for understanding context), etc.

papers_data = []
directory = '../abstracts/2000'

for file in os.listdir(directory):
    filename = os.fsdecode(file)
    with open(f'{directory}/{filename}') as f:
    
        text = f.read()
        
        scibert_data = getTitleAndAbstract(text)
        # print(scibert_data)

        embeddings = lda_embeddings(text)
        # print(embeddings)

        paper_id = re.search(r'Paper:\s*hep-th/(\d{7})', text)
        title = re.search(r'Title:\s*(.*)', text)
        subject = re.search(r'Subj-class:\s*(.*)', text) 

        papers_data.append({
            'paper_id': paper_id.group(1).strip(),
            'year': 2000,
            'title': title.group(1).strip() if title else "",
            'subject': subject.group(1).strip() if subject else "",
            'scibert_data': scibert_data,
            'embeddings': embeddings
        })


df_2000 = pd.DataFrame(papers_data)
df_2000.head()

,paper_id,year,title,subject,scibert_data,embeddings
0,0001001,2000,"Compactification, Geometry and Duality: N=2",,"[Compactification, Geometry and Duality: N=2. ...","[compactification, geometry, duality, note, ba..."
1,0001002,2000,Domain Walls and Massive Gauged Supergravity P...,,[Domain Walls and Massive Gauged Supergravity ...,"[domain, wall, massive, gauged, supergravity, ..."
2,0001003,2000,"Comment on ""Metric Fluctuations in Brane Worlds""",,"[Comment on ""Metric Fluctuations in Brane Worl...","[comment, metric, fluctuation, brane, world, r..."
3,0001004,2000,Moving Mirrors and Thermodynamic Paradoxes,,[Moving Mirrors and Thermodynamic Paradoxes. Q...,"[moving, mirror, thermodynamic, paradox, quant..."
4,0001005,2000,Bundles of chiral blocks and boundary conditio...,,[Bundles of chiral blocks and boundary conditi...,"[bundle, chiral, block, boundary, condition, c..."


In [47]:
df_2000.to_parquet('abstract structured data/papers2000.parquet')

In [48]:
# Storing Abstract data in structured format for different models 
# Ex: LDA (require processing), SciBERT (Entire sentences are required for understanding context), etc.

papers_data = []
directory = '../abstracts/2001'

for file in os.listdir(directory):
    filename = os.fsdecode(file)
    with open(f'{directory}/{filename}') as f:
    
        text = f.read()
        
        scibert_data = getTitleAndAbstract(text)
        # print(scibert_data)

        embeddings = lda_embeddings(text)
        # print(embeddings)

        paper_id = re.search(r'Paper:\s*hep-th/(\d{7})', text)
        title = re.search(r'Title:\s*(.*)', text)
        subject = re.search(r'Subj-class:\s*(.*)', text) 

        papers_data.append({
            'paper_id': paper_id.group(1).strip(),
            'year': 2001,
            'title': title.group(1).strip() if title else "",
            'subject': subject.group(1).strip() if subject else "",
            'scibert_data': scibert_data,
            'embeddings': embeddings
        })


df_2001 = pd.DataFrame(papers_data)
df_2001.head()

,paper_id,year,title,subject,scibert_data,embeddings
0,0101001,2001,On the Polarization of Unstable D0-Branes into...,,[On the Polarization of Unstable D0-Branes int...,"[polarization, unstable, branes, non, commutat..."
1,0101002,2001,BPS Vortices in Brane-Antibrane Effective Theory,,[BPS Vortices in Brane-Antibrane Effective The...,"[bps, vortex, brane, antibrane, effective, the..."
2,0101003,2001,String versus Einstein frame in an AdS/CFT ind...,,[String versus Einstein frame in an AdS/CFT in...,"[string, versus, einstein, frame, ad, cft, ind..."
3,0101004,2001,Black Hole Horizon in Superstring Inspired D-D...,,[Black Hole Horizon in Superstring Inspired D-...,"[black, hole, horizon, superstring, inspired, ..."
4,0101005,2001,Nonlocal two-dimensional Yang-Mills- and gener...,,[Nonlocal two-dimensional Yang-Mills- and gene...,"[nonlocal, two, dimensional, yang, mill, gener..."


In [49]:
df_2001.to_parquet('abstract structured data/papers2001.parquet')

In [50]:
# Storing Abstract data in structured format for different models 
# Ex: LDA (require processing), SciBERT (Entire sentences are required for understanding context), etc.

papers_data = []
directory = '../abstracts/2002'

for file in os.listdir(directory):
    filename = os.fsdecode(file)
    with open(f'{directory}/{filename}') as f:
    
        text = f.read()
        
        scibert_data = getTitleAndAbstract(text)
        # print(scibert_data)

        embeddings = lda_embeddings(text)
        # print(embeddings)

        paper_id = re.search(r'Paper:\s*hep-th/(\d{7})', text)
        title = re.search(r'Title:\s*(.*)', text)
        subject = re.search(r'Subj-class:\s*(.*)', text) 

        papers_data.append({
            'paper_id': paper_id.group(1).strip(),
            'year': 2002,
            'title': title.group(1).strip() if title else "",
            'subject': subject.group(1).strip() if subject else "",
            'scibert_data': scibert_data,
            'embeddings': embeddings
        })


df_2002 = pd.DataFrame(papers_data)
df_2002.head()

,paper_id,year,title,subject,scibert_data,embeddings
0,0201001,2002,Integrable Heisenberg-van Vleck chains with va...,,[Integrable Heisenberg-van Vleck chains with v...,"[integrable, heisenberg, van, vleck, chain, va..."
1,0201002,2002,"Non-Commutative Space-Times, Black Hole, and E...",,"[Non-Commutative Space-Times, Black Hole, and ...","[non, commutative, space, time, black, hole, e..."
2,0201003,2002,The Hamilton-Jacobi treatment for non-abelian ...,,[The Hamilton-Jacobi treatment for non-abelian...,"[hamilton, jacobi, treatment, non, abelian, ch..."
3,0201004,2002,Slicing and Brane Dependence of the (A)dS/CFT ...,,[Slicing and Brane Dependence of the (A)dS/CFT...,"[slicing, brane, dependence, cft, corresponden..."
4,0201005,2002,Vortex Correlation Functions in Maxwell-Chern-...,High Energy Physics - Theory; Strongly Correla...,[Vortex Correlation Functions in Maxwell-Chern...,"[vortex, correlation, function, maxwell, chern..."


In [51]:
df_2002.to_parquet('abstract structured data/papers2002.parquet')

In [52]:
# Storing Abstract data in structured format for different models 
# Ex: LDA (require processing), SciBERT (Entire sentences are required for understanding context), etc.

papers_data = []
directory = '../abstracts/2003'

for file in os.listdir(directory):
    filename = os.fsdecode(file)
    with open(f'{directory}/{filename}') as f:
    
        text = f.read()
        
        scibert_data = getTitleAndAbstract(text)
        # print(scibert_data)

        embeddings = lda_embeddings(text)
        # print(embeddings)

        paper_id = re.search(r'Paper:\s*hep-th/(\d{7})', text)
        title = re.search(r'Title:\s*(.*)', text)
        subject = re.search(r'Subj-class:\s*(.*)', text) 

        papers_data.append({
            'paper_id': paper_id.group(1).strip(),
            'year': 2003,
            'title': title.group(1).strip() if title else "",
            'subject': subject.group(1).strip() if subject else "",
            'scibert_data': scibert_data,
            'embeddings': embeddings
        })


df_2003 = pd.DataFrame(papers_data)
df_2003.head()

,paper_id,year,title,subject,scibert_data,embeddings
0,0301001,2003,Precanonical quantization of Yang-Mills fields...,,[Precanonical quantization of Yang-Mills field...,"[precanonical, quantization, yang, mill, field..."
1,0301002,2003,Universality classes for horizon instabilities,,[Universality classes for horizon instabilitie...,"[universality, class, horizon, instability, in..."
2,0301003,2003,Strings in the Quantized pp-wave Backgrounds f...,,[Strings in the Quantized pp-wave Backgrounds ...,"[string, quantized, wave, background, membrane..."
3,0301004,2003,Relativistic phase space: dimensional recurrences,,[Relativistic phase space: dimensional recurre...,"[relativistic, phase, space, dimensional, recu..."
4,0301005,2003,Structure of supergravity theories,,[Structure of supergravity theories. We give a...,"[structure, supergravity, theory, give, elemen..."


In [53]:
df_2003.to_parquet('abstract structured data/papers2003.parquet')

# Abstract Analysis

In [ ]:
# Combining all the extracted paper information (parquet files) into a single file for analysis.

import pandas as pd
from pathlib import Path

DATA_DIR = Path("./abstract structured data")
years_needed = range(1992, 2004)  # include 1992 in case any paper_id in your graph traces back that far

paper_dfs = []
for year in years_needed:
    path = DATA_DIR / f"papers{year}.parquet"
    if path.exists():
        df = pd.read_parquet(path, columns=["paper_id", "year", "title", "subject", "embeddings", "scibert_data"])
        paper_dfs.append(df)
    else:
        print(f"Missing file: {path}")

papers_all = pd.concat(paper_dfs, ignore_index=True)
print(f"Loaded {len(papers_all)} papers from {len(paper_dfs)} files")

Loaded 29555 papers from 12 files


In [ ]:
# papers_all.head()

In [ ]:
# Checking how many papers have a subject line

# coverage = papers_all["subject"].notna().mean()
# print(f"{coverage:.1%} of papers have a non-null subject tag")
# if coverage > 0:
#     print(papers_all["subject"].value_counts().head(20))

In [ ]:
# Saving scibert embeddings (Title + Abstract of a paper) to a proper file.

# from sentence_transformers import SentenceTransformer
# import numpy as np

# model = SentenceTransformer("allenai/specter2_base")  # purpose-built for scientific papers; swap for a SciBERT sentence-transformer variant if you specifically need SciBERT

# def extract_text(scibert_data):
#     # unwrap the list -- adjust if scibert_data sometimes contains more than one element
#     if isinstance(scibert_data, list) and len(scibert_data) > 0:
#         return scibert_data[0]
#     return str(scibert_data)

# paper_ids_all = papers_all["paper_id"].tolist()
# texts_all = papers_all["scibert_data"].apply(extract_text).tolist()

# print(f"Encoding {len(texts_all)} papers...")
# embeddings_all = model.encode(texts_all, batch_size=64, show_progress_bar=True)

# paper_id_to_embedding = dict(zip(paper_ids_all, embeddings_all))

In [3]:
# with open("paper_id_to_embedding.pkl", "wb") as f:
#     pickle.dump(paper_id_to_embedding, f)

# Loading the extracted scibert data:
with open("paper_id_to_embedding.pkl", "rb") as f:
    paper_id_to_embedding = pickle.load(f)

In [12]:
paper_id_to_tokens = papers_all.set_index("paper_id")["embeddings"].to_dict()   # for TF-IDF labeling

The papers present in the final year of a communities timeline will give a reasonable domain or description of the community, hence get papers from the final stage (or the most recent year of the community) and then extract information from the papers abstract data.

In [5]:
# loading persistent communities and slice partitions for the abstract analysis code.

with open("persistent_communities.pkl", "rb") as f:
    persistent_communities = pickle.load(f)

with open("slice_partitions.pkl", "rb") as f:
    slice_partitions = pickle.load(f)

print(f"{len(persistent_communities)} persistent communities loaded")
print(f"Years available in slice_partitions: {list(slice_partitions.keys())}")

17 persistent communities loaded
Years available in slice_partitions: [1993, 1995, 1997, 1999, 2001, 2003]


In [7]:
def get_terminal_snapshot_papers(global_label, persistent_communities, slice_partitions):
    info = persistent_communities[global_label]
    final_year, final_local_id, final_size, final_status = info["timeline"][-1]
    return final_year, set(slice_partitions[final_year][final_local_id])

community_labels_to_test = [2, 4, 16, 17, 25, 27, 29, 40, 43, 45, 48, 49, 50, 52, 53, 55, 56]

In [8]:
def extract_text(scibert_data):
    if isinstance(scibert_data, list) and len(scibert_data) > 0:
        return scibert_data[0]
    return str(scibert_data)

paper_id_to_title = papers_all.set_index("paper_id")["title"].to_dict()

paper_id_to_fulltext = {
    pid: extract_text(sd)
    for pid, sd in papers_all.set_index("paper_id")["scibert_data"].to_dict().items()
}

In [9]:
# TOPICBert analysis

# pip install bertopic --quiet

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.dimensionality import BaseDimensionalityReduction
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
import numpy as np

class BaseCluster:
    """Dummy clusterer — just passes through pre-assigned labels, does no clustering."""
    def fit(self, X):
        self.labels_ = X
        return self
    def predict(self, X):
        return X

def run_bertopic_labeling(community_labels, persistent_communities, slice_partitions,
                           paper_id_to_title, paper_id_to_embedding, top_n=10):
    """
    paper_id_to_title: dict {paper_id: raw title/abstract text}
    paper_id_to_embedding: dict {paper_id: 1D numpy array} -- your existing SciBERT embeddings,
                            computed on raw unprocessed sentences
    """
    docs = []
    embeddings = []
    y = []  # cluster assignment per doc, using global_label as the class id
    community_meta = {}

    for global_label in community_labels:
        final_year, paper_ids = get_terminal_snapshot_papers(global_label, persistent_communities, slice_partitions)
        community_meta[global_label] = final_year
        for pid in paper_ids:
            if pid in paper_id_to_title and pid in paper_id_to_embedding:
                docs.append(paper_id_to_fulltext[pid])
                embeddings.append(paper_id_to_embedding[pid])
                y.append(global_label)

    embeddings = np.vstack(embeddings)
    y = np.array(y)

    empty_dimensionality_model = BaseDimensionalityReduction()
    empty_cluster_model = BaseCluster()
    ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)  # down-weights corpus-wide filler, same intent as your sublinear_tf fix

    academic_filler = {
        "show", "shows", "shown", "obtain", "obtains", "obtained", "study", "studied",
        "consider", "considered", "present", "presented", "discuss", "propose", "proposed",
        "demonstrate", "demonstrated", "derive", "derived", "find", "finding", "findings",
        "yield", "yields", "yielded", "reproduce", "reproduces", "paper", "abstract",
        "introduction", "conclusion", "result", "results", "case", "cases", "using", "used",
        "based", "new", "known", "recent", "recently", "possible", "possibly", "general",
        "generally", "review", "approach", "method", "methods"
    }

    custom_stopwords = list(ENGLISH_STOP_WORDS.union(academic_filler))
    vectorizer_model = CountVectorizer(stop_words=custom_stopwords, min_df=2)

    topic_model = BERTopic(
        hdbscan_model=empty_cluster_model,
        umap_model=empty_dimensionality_model,
        vectorizer_model= vectorizer_model,
        ctfidf_model=ctfidf_model,
        verbose=False
    )

    topics, probs = topic_model.fit_transform(docs, embeddings=embeddings, y=y)

    results = {}
    for global_label in community_labels:
        try:
            top_words = topic_model.get_topic(global_label)  # list of (word, score) tuples
            results[global_label] = top_words[:top_n]
        except Exception:
            results[global_label] = None

    return results, topic_model

bertopic_results, topic_model = run_bertopic_labeling(
    community_labels_to_test, persistent_communities, slice_partitions,
    paper_id_to_title, paper_id_to_embedding
)

for label, terms in bertopic_results.items():
    print(f"Community {label}: {terms}")

d:\Anaconda\envs\capstone\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Community 2: [('secondquantization', np.float64(0.4677901514760513)), ('constituting', np.float64(0.39577263245988487)), ('secondquantized', np.float64(0.2752525277000497)), ('d5branes', np.float64(0.1897159783065151)), ('d5brane', np.float64(0.1301559789652734)), ('overlapping', np.float64(0.1229864165524591)), ('caused', np.float64(0.11660134391019362)), ('cycles', np.float64(0.11389977913523056)), ('resolve', np.float64(0.10822464898162297)), ('subspace', np.float64(0.07185984038197338))]
Community 4: [('evasion', np.float64(0.3350459419736783)), ('supercovariantly', np.float64(0.2954048792622154)), ('guaranteed', np.float64(0.16476877794418618)), ('solvability', np.float64(0.11182159426786223)), ('nogo', np.float64(0.10859848924251411)), ('saturate', np.float64(0.0836598930788135)), ('bounds', np.float64(0.080434231141791)), ('conical', np.float64(0.06432339065529862)), ('self', np.float64(0.05377184809000247)), ('bogomolnyi', np.float64(0.046260551914056564))]
Community 16: [('ord

In [13]:
# TF-ICF methodlogy mentioned in Greene 2025 paper.

from collections import Counter
import math

def compute_icf_for_year(slice_year, slice_partitions, paper_id_to_tokens):
    """ICF(t_k) = log(|C| / CF(t_k)), where |C| = number of step communities in this year,
    CF(t_k) = number of those communities containing term t_k at all."""
    communities = slice_partitions[slice_year]
    n_communities = len(communities)

    community_freq = Counter()
    for community in communities:
        terms_in_this_community = set()
        for pid in community:
            terms_in_this_community.update(paper_id_to_tokens.get(pid, []))
        community_freq.update(terms_in_this_community)  # each term counted once per community it appears in

    icf = {term: math.log(n_communities / cf) for term, cf in community_freq.items() if cf > 0}
    return icf

def compute_tf_idf_within_community(paper_ids, paper_id_to_tokens):
    """TF(t_k, C_i): raw term count within the community.
    IDF(t_k, C_i) = log(|C_i| / DF(t_k, C_i)) -- DF here is per-paper document frequency
    WITHIN this single community (i.e., how many papers in C_i contain the term)."""
    n_papers = len(paper_ids)
    term_counts = Counter()
    doc_freq = Counter()

    for pid in paper_ids:
        tokens = paper_id_to_tokens.get(pid, [])
        term_counts.update(tokens)
        doc_freq.update(set(tokens))

    idf_within = {term: math.log(n_papers / df) for term, df in doc_freq.items() if df > 0}
    return term_counts, idf_within

def tf_icf_label(global_label, persistent_communities, slice_partitions, paper_id_to_tokens, top_n=10):
    final_year, paper_ids = get_terminal_snapshot_papers(global_label, persistent_communities, slice_partitions)

    icf = compute_icf_for_year(final_year, slice_partitions, paper_id_to_tokens)
    tf, idf_within = compute_tf_idf_within_community(paper_ids, paper_id_to_tokens)

    scores = {}
    for term, tf_count in tf.items():
        if term in icf and term in idf_within:
            scores[term] = tf_count * idf_within[term] * icf[term]

    ranked = sorted(scores.items(), key=lambda x: -x[1])[:top_n]
    return final_year, len(paper_ids), ranked

tf_icf_results = {}
for global_label in community_labels_to_test:
    final_year, n_papers, ranked = tf_icf_label(global_label, persistent_communities, slice_partitions, paper_id_to_tokens)
    tf_icf_results[global_label] = ranked
    print(f"Community {global_label} (year {final_year}, n={n_papers}): {ranked}")

Community 2 (year 2003, n=741): [('bmn', 497.25944020559655), ('penrose', 455.63498688649776), ('wznw', 313.8480188301057), ('wzw', 259.656705115291), ('duality', 234.8915002638799), ('branes', 227.79991170542652), ('ad', 214.6286683808482), ('icft', 199.06760360992646), ('wave', 198.78311952846113), ('plane', 187.56828963833965)]
Community 4 (year 2003, n=1647): [('radion', 1149.8905488279759), ('brane', 911.8937924817373), ('randall', 779.0623446532265), ('sundrum', 762.533429905872), ('braneworld', 640.3979758343371), ('wall', 605.1345173877987), ('bulk', 577.7810376358173), ('ekpyrotic', 562.7676686730838), ('black', 553.2365469615721), ('branes', 524.7859478354324)]
Community 16 (year 2003, n=2035): [('rsos', 1089.3747151628686), ('tba', 869.2336129313829), ('toda', 799.8626198907256), ('xxz', 775.0341998623168), ('bethe', 703.350099860965), ('affine', 605.6926652653514), ('boundary', 594.3337038401517), ('potts', 524.6042210567863), ('integrable', 510.5203706042876), ('chain', 48

## The information for labelling was all extracted using the outputs from both TOPICBert and TF-ICF. 

In [14]:
for label in community_labels_to_test:
    print(f"\n=== Community {label} ===")
    print(f"  BERTopic:  {[w for w, s in (bertopic_results.get(label) or [])][:8]}")
    print(f"  TF-ICF:    {[w for w, s in tf_icf_results.get(label, [])][:8]}")
    # print(f"  Your earlier corpus-IDF TF-IDF: {[w for w, s in community_field_labels.get(label, [])][:8]}")


=== Community 2 ===
  BERTopic:  ['secondquantization', 'constituting', 'secondquantized', 'd5branes', 'd5brane', 'overlapping', 'caused', 'cycles']
  TF-ICF:    ['bmn', 'penrose', 'wznw', 'wzw', 'duality', 'branes', 'ad', 'icft']

=== Community 4 ===
  BERTopic:  ['evasion', 'supercovariantly', 'guaranteed', 'solvability', 'nogo', 'saturate', 'bounds', 'conical']
  TF-ICF:    ['radion', 'brane', 'randall', 'sundrum', 'braneworld', 'wall', 'bulk', 'ekpyrotic']

=== Community 16 ===
  BERTopic:  ['ordinarily', 'annihilated', 'virtue', 'equality', 'bosefermi', 'ensure', 'surprising', 'fashion']
  TF-ICF:    ['rsos', 'tba', 'toda', 'xxz', 'bethe', 'affine', 'boundary', 'potts']

=== Community 17 ===
  BERTopic:  ['explained', 'nonsupersymmetric', 'absence', 'vanishing', 'strong', 'vacuum', 'cosmological', 'massless']
  TF-ICF:    ['orientifolds', 'yau', 'calabi', 'orientifold', 'compactifications', 'branes', 'iia', 'heterotic']

=== Community 25 ===
  BERTopic:  ['curvedspace', 'virtue',

## Extracting year wise icf results

This step was performed to get an overall idea of papers throughout the timelines of each community, so as to see the domains in which papers were working on per year basis or research domain shifts per year.

In [15]:
icf_cache = {}

def get_icf_for_year_cached(slice_year, slice_partitions, paper_id_to_tokens):
    if slice_year not in icf_cache:
        icf_cache[slice_year] = compute_icf_for_year(slice_year, slice_partitions, paper_id_to_tokens)
    return icf_cache[slice_year]

In [16]:
from collections import Counter

corpus_doc_freq = Counter()
for tokens in paper_id_to_tokens.values():
    corpus_doc_freq.update(set(tokens))

In [17]:
def community_evolution_tficf(global_label, persistent_communities, slice_partitions, paper_id_to_tokens, top_n=5, min_corpus_df=3, corpus_doc_freq=None):
    info = persistent_communities[global_label]
    evolution = []

    for slice_year, local_id, size, status in info["timeline"]:
        paper_ids = slice_partitions[slice_year][local_id]

        icf = get_icf_for_year_cached(slice_year, slice_partitions, paper_id_to_tokens)
        tf, idf_within = compute_tf_idf_within_community(paper_ids, paper_id_to_tokens)

        scores = {}
        for term, tf_count in tf.items():
            if term in icf and term in idf_within:
                if corpus_doc_freq is not None and corpus_doc_freq.get(term, 0) < min_corpus_df:
                    continue
                scores[term] = tf_count * idf_within[term] * icf[term]

        top_terms = sorted(scores.items(), key=lambda x: -x[1])[:top_n]
        evolution.append((slice_year, size, status, top_terms))

    return evolution

In [18]:
for label in persistent_communities:
    print(f"\n=== Community {label} ===")
    for slice_year, size, status, top_terms in community_evolution_tficf(
        label, persistent_communities, slice_partitions, paper_id_to_tokens,
        min_corpus_df=3, corpus_doc_freq=corpus_doc_freq
    ):
        terms_str = ", ".join(t for t, s in top_terms)
        print(f"  {slice_year} (size {size}, {status}): {terms_str}")


=== Community 2 ===
  1993 (size 125, birth): wznw, isometry, black, gauging, wzw
  1995 (size 401, growth): black, hole, wznw, dilaton, sitter
  1997 (size 601, growth): bang, graceful, exit, wznw, black
  1999 (size 444, shrink): wznw, icft, wzw, eikonal, duality
  2001 (size 204, shrink): poisson, nonhomogeneous, wznw, drinfeld, monodromy
  2003 (size 741, growth): bmn, penrose, wznw, wzw, duality

=== Community 4 ===
  1993 (size 101, birth): antifield, antibracket, batalin, vilkovisky, transport
  1995 (size 287, growth): antifield, antibracket, antifields, triplectic, batalin
  1997 (size 521, growth): antibracket, antifield, antifields, triplectic, antibrackets
  1999 (size 9, shrink): jet, finsler, superbundles, superspaces, prolongation
  2001 (size 1409, growth): brane, radion, randall, sundrum, braneworld
  2003 (size 1647, growth): radion, brane, randall, sundrum, braneworld

=== Community 16 ===
  1993 (size 52, birth): dilogarithm, rogers, resonance, ramanujan, tba
  199